<a href="https://colab.research.google.com/github/GKSJ-AI-CliniScan/MedAssistAI/blob/TahuraShaikh/LightGBM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import time
import warnings

warnings.filterwarnings("ignore")

# Load dataset
dataset = pd.read_csv("/content/dataset2_training.csv")

print("Dataset Shape:", dataset.shape)
print("Number of Features:", dataset.shape[1] - 1)
print("Number of Diseases:", dataset["Disease"].nunique())
print("\nFirst 5 rows:")
display(dataset.head())

Dataset Shape: (60000, 378)
Number of Features: 377
Number of Diseases: 658

First 5 rows:


,Disease,anxiety and nervousness,depression,shortness of breath,depressive or psychotic symptoms,sharp chest pain,dizziness,insomnia,abnormal involuntary movements,chest tightness,...,stuttering or stammering,problems with orgasm,nose deformity,lump over jaw,sore in nose,hip weakness,back swelling,ankle stiffness or tightness,ankle weakness,neck weakness
0,Coronary Atherosclerosis,0,0,1,0,1,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
1,Cholecystitis,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,Osteoarthritis,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,Vaginal Yeast Infection,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,Concussion,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [2]:
disease_counts = dataset["Disease"].value_counts()

print("Total Diseases:", len(disease_counts))
print("Minimum samples per disease:", disease_counts.min())
print("Maximum samples per disease:", disease_counts.max())

print("\nDisease Distribution Summary:")
print(disease_counts.describe())

print("\nDiseases with fewer than 5 samples:")
print(disease_counts[disease_counts < 5])

Total Diseases: 658
Minimum samples per disease: 2
Maximum samples per disease: 386

Disease Distribution Summary:
count    658.000000
mean      91.185410
std      109.099227
min        2.000000
25%        8.000000
50%       47.000000
75%      156.000000
max      386.000000
Name: count, dtype: float64

Diseases with fewer than 5 samples:
Disease
Urethral Valves             4
Ectropion                   4
Injury To The Abdomen       4
Fracture Of The Foot        4
Open Wound Of The Lip       4
                           ..
Pseudohypoparathyroidism    2
Pinguecula                  2
Syringomyelia               2
Conversion Disorder         2
Volvulus                    2
Name: count, Length: 112, dtype: int64


In [3]:
X = dataset.drop(columns=["Disease"])
y = dataset["Disease"]

print("X Shape:", X.shape)
print("y Shape:", y.shape)
print("Unique Diseases:", y.nunique())

X Shape: (60000, 377)
y Shape: (60000,)
Unique Diseases: 658


In [4]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(y)

print("Number of Classes:", len(label_encoder.classes_))
print("First 10 Original Labels:")
print(label_encoder.classes_[:10])

print("\nFirst 10 Encoded Labels:")
print(y_encoded[:10])

Number of Classes: 658
First 10 Original Labels:
['Abdominal Aortic Aneurysm' 'Abdominal Hernia' 'Abscess Of Nose'
 'Abscess Of The Lung' 'Abscess Of The Pharynx' 'Acanthosis Nigricans'
 'Acariasis' 'Achalasia' 'Acne' 'Actinic Keratosis']

First 10 Encoded Labels:
[143 105 437 635 129 457  50 164 535 125]


In [5]:
print("Classes with only 1 sample:", (disease_counts == 1).sum())
print("Classes with only 2 samples:", (disease_counts == 2).sum())

Classes with only 1 sample: 0
Classes with only 2 samples: 44


In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.20,
    random_state=42,
    stratify=y_encoded
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

print("Unique classes in full dataset :", len(np.unique(y_encoded)))
print("Unique classes in training     :", len(np.unique(y_train)))
print("Unique classes in testing      :", len(np.unique(y_test)))

X_train: (48000, 377)
X_test : (12000, 377)
Unique classes in full dataset : 658
Unique classes in training     : 658
Unique classes in testing      : 614


In [7]:
train_counts = pd.Series(y_train).value_counts()
test_counts = pd.Series(y_test).value_counts()

print("Training class count summary:")
print(train_counts.describe())

print("\nTest class count summary:")
print(test_counts.describe())

print("\nTraining classes with <= 2 samples:")
print((train_counts <= 2).sum())

print("Training classes with <= 5 samples:")
print((train_counts <= 5).sum())

print("\nTest classes with <= 2 samples:")
print((test_counts <= 2).sum())

Training class count summary:
count    658.000000
mean      72.948328
std       87.266141
min        2.000000
25%        6.000000
50%       38.000000
75%      125.000000
max      309.000000
Name: count, dtype: float64

Test class count summary:
count    614.000000
mean      19.543974
std       22.032540
min        1.000000
25%        2.000000
50%       10.000000
75%       33.000000
max       77.000000
Name: count, dtype: float64

Training classes with <= 2 samples:
85
Training classes with <= 5 samples:
149

Test classes with <= 2 samples:
171


In [8]:
print(X.dtypes.value_counts())
print("\nMissing values:", X.isnull().sum().sum())

int64    377
Name: count, dtype: int64

Missing values: 0


In [9]:
!pip install -q lightgbm

In [10]:
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [11]:
# Create a copy so the original X remains unchanged
X_lgb = X.copy()

# Rename all feature columns to simple safe names
X_lgb.columns = [f"feature_{i}" for i in range(X_lgb.shape[1])]

print("Number of features:", X_lgb.shape[1])
print("First 10 feature names:")
print(X_lgb.columns[:10].tolist())

Number of features: 377
First 10 feature names:
['feature_0', 'feature_1', 'feature_2', 'feature_3', 'feature_4', 'feature_5', 'feature_6', 'feature_7', 'feature_8', 'feature_9']


In [12]:
X_train_lgb, X_test_lgb, y_train_lgb, y_test_lgb = train_test_split(
    X_lgb,
    y_encoded,
    test_size=0.20,
    random_state=42,
    stratify=y_encoded
)

print("X_train_lgb:", X_train_lgb.shape)
print("X_test_lgb :", X_test_lgb.shape)

print("Train classes:", len(np.unique(y_train_lgb)))
print("Test classes :", len(np.unique(y_test_lgb)))

X_train_lgb: (48000, 377)
X_test_lgb : (12000, 377)
Train classes: 658
Test classes : 614


In [13]:
from lightgbm import LGBMClassifier

start = time.time()

lgb = LGBMClassifier(
    objective="multiclass",
    num_class=658,
    n_estimators=100,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=10,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

lgb.fit(X_train_lgb, y_train_lgb)

end = time.time()

print("LightGBM Training Time:", round(end - start, 2), "seconds")

LightGBM Training Time: 255.67 seconds


In [14]:
start = time.time()

lgb_pred = lgb.predict(X_test_lgb)

end = time.time()

print("Prediction Time:", round(end - start, 2), "seconds")
print("Unique predictions:", len(np.unique(lgb_pred)))
print("First 20 predictions:", np.unique(lgb_pred)[:20])

Prediction Time: 5.75 seconds
Unique predictions: 29
First 20 predictions: [ 10  19  24  39  78 129 140 156 161 175 187 208 255 267 300 324 341 347
 357 367]


In [15]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

lgb_accuracy = accuracy_score(y_test_lgb, lgb_pred)

lgb_precision = precision_score(
    y_test_lgb,
    lgb_pred,
    average="weighted",
    zero_division=0
)

lgb_recall = recall_score(
    y_test_lgb,
    lgb_pred,
    average="weighted",
    zero_division=0
)

lgb_f1 = f1_score(
    y_test_lgb,
    lgb_pred,
    average="weighted",
    zero_division=0
)

print("LightGBM Results")
print("----------------")
print("Accuracy :", round(lgb_accuracy, 4))
print("Precision:", round(lgb_precision, 4))
print("Recall   :", round(lgb_recall, 4))
print("F1 Score :", round(lgb_f1, 4))

LightGBM Results
----------------
Accuracy : 0.0088
Precision: 0.0159
Recall   : 0.0088
F1 Score : 0.003


In [16]:
# Check the probability predictions
lgb_proba = lgb.predict_proba(X_test_lgb)

print("Probability shape:", lgb_proba.shape)
print("Expected classes:", len(lgb.classes_))
print("Number of unique predicted classes:", len(np.unique(lgb_pred)))

# Check the highest probability for each test sample
max_probs = np.max(lgb_proba, axis=1)

print("\nMaximum probability statistics:")
print(pd.Series(max_probs).describe())

Probability shape: (12000, 658)
Expected classes: 658
Number of unique predicted classes: 29

Maximum probability statistics:
count    12000.000000
mean         0.999959
std          0.004242
min          0.535853
25%          1.000000
50%          1.000000
75%          1.000000
max          1.000000
dtype: float64


In [17]:
# Count how often each class is predicted
prediction_counts = pd.Series(lgb_pred).value_counts()

print("Number of predicted classes:", len(prediction_counts))
print("\nMost frequently predicted classes:")
print(prediction_counts.head(20))

Number of predicted classes: 29

Most frequently predicted classes:
420    11191
399      266
175      183
381       72
187       68
208       51
376       26
129       25
19        25
161       18
300       18
527       10
156        7
357        7
39         4
613        4
140        4
10         3
24         3
579        3
Name: count, dtype: int64


In [18]:
from sklearn.utils.class_weight import compute_sample_weight

# Calculate weights based on the training labels
sample_weights = compute_sample_weight(
    class_weight="balanced",
    y=y_train_lgb
)

print("Sample weights created")
print("Minimum weight:", sample_weights.min())
print("Maximum weight:", sample_weights.max())

Sample weights created
Minimum weight: 0.23607873225720777
Maximum weight: 36.474164133738604


In [19]:
start = time.time()

lgb_balanced = LGBMClassifier(
    objective="multiclass",
    num_class=658,
    n_estimators=100,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=10,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

lgb_balanced.fit(
    X_train_lgb,
    y_train_lgb,
    sample_weight=sample_weights
)

end = time.time()

print("LightGBM Balanced Training Time:",
      round(end - start, 2), "seconds")

LightGBM Balanced Training Time: 295.84 seconds


In [20]:
start = time.time()

lgb_balanced_pred = lgb_balanced.predict(X_test_lgb)

end = time.time()

print("Prediction Time:", round(end - start, 2), "seconds")
print("Unique predictions:",
      len(np.unique(lgb_balanced_pred)))

Prediction Time: 30.42 seconds
Unique predictions: 288


In [21]:
lgb_balanced_accuracy = accuracy_score(
    y_test_lgb,
    lgb_balanced_pred
)

lgb_balanced_precision = precision_score(
    y_test_lgb,
    lgb_balanced_pred,
    average="weighted",
    zero_division=0
)

lgb_balanced_recall = recall_score(
    y_test_lgb,
    lgb_balanced_pred,
    average="weighted",
    zero_division=0
)

lgb_balanced_f1 = f1_score(
    y_test_lgb,
    lgb_balanced_pred,
    average="weighted",
    zero_division=0
)

print("LightGBM Balanced Results")
print("-------------------------")
print("Accuracy :", round(lgb_balanced_accuracy, 4))
print("Precision:", round(lgb_balanced_precision, 4))
print("Recall   :", round(lgb_balanced_recall, 4))
print("F1 Score :", round(lgb_balanced_f1, 4))

LightGBM Balanced Results
-------------------------
Accuracy : 0.0589
Precision: 0.1306
Recall   : 0.0589
F1 Score : 0.0744
